# Musanga V3 / 02D — Rotating-circle CV × illumination correction

**Primary:** three spatial folds from Notebook 01. All 22 complete circles are tested exactly once. For each fold, train M1 (partial crowns only), M2 (partial crowns + training-fold exhaustive circles), and calibrate M1 into M3 with training-fold circles. Compare original RGB and exploratory brightness-harmonized RGB; **6 model variants**, identical reference circles and metrics. MOS treated as one forest. This harmonization is a controlled test, **not a claim to reproduce V2 Notebook 03**. Training/validation are geographically buffered. No satellite inference is performed here.

**Expected results:** held-out circle scatter for six variants, R², slope/intercept, MAE, RMSE and bias. Fold outputs and checkpoint files saved incrementally. Full-circle inference can take longer than GPU training.


## 01 — Setup


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
BASE_OUT=ROOT/'Outputs/V3/02_rotating_cv';BASE_OUT.mkdir(parents=True,exist_ok=True)
OUT=BASE_OUT
REF=ROOT/'Outputs/V3/01_reference_data'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
FOLDS=[0,1,2];MODES=['native','harmonized']
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)





## 02 — Source geometries and model architecture


In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>.05 or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')




## 03 — Reuse original partial-only and combined crown masks; reset fold-specific reference circles


In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
all_circle_buffers={}
BLOCKED={}
DEV=set();HOLDOUT=set()
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs')]
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(series,name,win):
    # GeoPandas spatial index avoids scanning every tree polygon for every crop.
    # rasterize clips geometry to the requested window automatically.
    d=geo[name]
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    if len(ids)==0:
        return np.zeros((int(win.height),int(win.width)),dtype=bool)
    gs=series.iloc[ids]
    return rasterize(((geom,1) for geom in gs if geom is not None and not geom.is_empty),
       out_shape=(int(win.height),int(win.width)),
       transform=rasterio.windows.transform(win,d['transform']),
       fill=0,dtype='uint8').astype(bool)

def rgb_for_window(name,win):
    with rasterio.open(geo[name]['path']) as ds:
        raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid

def training_crop(rec,model):
    name=rec.mosaic;d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    t=d['trees_local']
    pos_tree=burn_window(t.loc[t.class01==1].geometry,name,win)
    neg_tree=burn_window(t.loc[t.class01==0].geometry,name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def make_training_cache(model):
    selected=[]
    for (site,kind),grp in candidate.groupby(['site','kind']):
        if model=='M1' and kind=='exhaustive':continue
        if model=='M2' and kind=='exhaustive':
            grp=grp[grp.circle_id.isin(DEV)]
        limit=MAX_EXHAUSTIVE_PER_SITE if kind=='exhaustive' else MAX_PARTIAL_PER_MOSAIC*(3 if site=='MOS' else 1)
        subset=grp.sample(n=min(limit,len(grp)),random_state=SEED)
        selected.extend(subset.itertuples(index=False))
    result=[]
    prep_started=time.perf_counter()
    for i,rec in enumerate(selected):
        chip=training_crop(rec,model)
        if chip is not None:result.append(chip)
        if (i+1)%100==0:
            print(f"{model}: prepared {i+1}/{len(selected)} candidates, usable {len(result)}, elapsed {time.perf_counter()-prep_started:.1f}s",flush=True)
    site_counts = pd.Series([item["site"] for item in result]).value_counts().to_dict()
    print(f"{model}: usable training chips = {len(result)}", flush=True)
    print("Training chips by site:", site_counts, flush=True)
    print(f"{model}: cache preparation took {time.perf_counter()-prep_started:.1f}s",flush=True)
    assert len(result)>15,f'{model}: too few chips'
    return result

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))


def configure_fold(fold):
    global FOLD,DEV,HOLDOUT,BLOCKED,split_df
    FOLD=int(fold)
    HOLDOUT=set(links.loc[links.fold==fold,'circle_id'].astype(int))
    DEV=set(links.circle_id.astype(int))-HOLDOUT
    assert len(HOLDOUT)>0 and len(DEV)>0
    split_df=links[['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']=np.where(split_df.circle_id.isin(HOLDOUT),'holdout','development')
    BLOCKED={}
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
        group_ids=set(links[(links.site==d['site'])&links.circle_id.isin(HOLDOUT)].circle_id.astype(int))
        BLOCKED[name]=[g.buffer(15.) for g in cs[cs.original_row.isin(group_ids)].geometry]
    assert split_df.query("split=='holdout'").site.nunique()==4
    print('Fold',fold,'holdout',sorted(HOLDOUT),'training/calibration circles',len(DEV),flush=True)


## 04 — Exploratory illumination harmonization

A reproducible **per-mosaic brightness-gain correction** derived only from unlabelled RGB samples. Mosaic medians are adjusted toward the pooled median, with gains capped from 0.7 to 1.4. This is a modest harmonization control, not V2's unverified correction method. Holdout labels are never consulted; target-image RGB is allowed (we are mapping existing orthomosaics).


In [ ]:
from rasterio.windows import Window
BASE_RGB_FUNCTION=rgb_for_window
RGB_MODE='native'
def estimate_mosaic_luminance(name,steps=8,size=160):
    with rasterio.open(geo[name]['path']) as ds:
        samples=[]
        for row in np.linspace(0,max(0,ds.height-size),steps).astype(int):
            for col in np.linspace(0,max(0,ds.width-size),steps).astype(int):
                win=Window(int(col),int(row),min(size,ds.width),min(size,ds.height))
                a=ds.read([1,2,3],window=win,masked=True).filled(0).astype('float32')/255
                lum=(.2126*a[0]+.7152*a[1]+.0722*a[2])
                ok=(lum>.035)&(lum<.96)
                if ok.sum()>100:samples.append(np.median(lum[ok]))
    assert samples,f'No usable luminance samples in {name}'
    return float(np.median(samples))
LUM={name:estimate_mosaic_luminance(name) for name in geo}
TARGET_LUM=float(np.median(list(LUM.values())))
GAIN={name:float(np.clip(TARGET_LUM/value,.7,1.4)) for name,value in LUM.items()}
display(pd.DataFrame([dict(mosaic=k,luminance=v,brightness_gain=GAIN[k]) for k,v in LUM.items()]))
(BASE_OUT/'illumination_gains.json').write_text(json.dumps(dict(reference=TARGET_LUM,luminance=LUM,gains=GAIN),indent=2))
def rgb_for_window(name,win):
    x,valid=BASE_RGB_FUNCTION(name,win)
    if RGB_MODE=='harmonized':
        x=np.clip(x*GAIN[name],0,1).astype('float32')
    return x,valid


## 05 — Train M1/M2 using the existing network; checkpoint each fold


In [ ]:
import time
def fit_network(label):
    ckpt=OUT/f'{label}_model.pt'
    torch.manual_seed(SEED)
    model=TinyUNet().to(device)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
        print('Loaded',ckpt,flush=True)
        return model
    print(f"{label}: building training cache...",flush=True)
    cache=make_training_cache(label)
    print(f"{label}: starting GPU training, {EPOCHS} epochs, {len(cache)} chips",flush=True)
    loader=DataLoader(CropDataset(cache),batch_size=BATCH,shuffle=True,num_workers=0)
    optim=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=1e-4)
    history=[]
    for epoch in range(EPOCHS):
        model.train();losses=[];start=time.time()
        for x,y,w in loader:
            x=x.to(device);y=y.to(device);w=w.to(device)
            optim.zero_grad(set_to_none=True)
            loss=masked_loss(model(x),y,w)
            loss.backward();optim.step()
            losses.append(float(loss.item()))
        history.append(dict(epoch=epoch+1,masked_BCE=float(np.mean(losses))))
        pd.DataFrame(history).to_csv(OUT/f'{label}_loss.csv',index=False)
        print(f'{label}: epoch {epoch+1}/{EPOCHS} BCE={np.mean(losses):.4f}, {time.time()-start:.1f}s',flush=True)
    torch.save(model.state_dict(),ckpt)
    del cache
    return model




## 06 — Complete-circle inference and calibration


In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_circle_raw.csv'
    histfile=OUT/f'{model_name}_histograms.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df.iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            rgb,valid=rgb_for_window(name,win)
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert len(frame)==22
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms



In [ ]:
def calibrated_bin_probs(theta):
    a,b=np.asarray(theta,dtype=float)
    return expit(a*logit(np.clip(BIN_CENTER,1e-5,1-1e-5))+b)



## 07 — Execute/resume folds and RGB treatments

M1 excludes all 22 circles from training, regardless of fold; M2 excludes the active fold's held-out circles. M3 only calibrates M1 using other-fold development circles. A checkpoint and score CSV is saved per model/variant/fold. As each circle belongs to one fold, final metrics have **22 independent circle-level test points** per variant.


In [ ]:
import gc
all_fold_scores=[]
for mode in MODES:
    RGB_MODE=mode
    for fold in FOLDS:
        configure_fold(fold)
        OUT=BASE_OUT/mode/f'fold_{fold}'
        OUT.mkdir(parents=True,exist_ok=True)
        if (OUT/'cv_holdout_scores.csv').exists():
            print('SKIP complete',mode,fold,flush=True)
            all_fold_scores.append(pd.read_csv(OUT/'cv_holdout_scores.csv'))
            continue
        print('===',mode,'fold',fold,'===',flush=True)
        model1=fit_network('M1')
        model2=fit_network('M2')
        m1,hist1=run_all_circles(model1,'M1')
        m2,hist2=run_all_circles(model2,'M2')
        dev_table=m1[m1.split=='development']
        assert dev_table.circle_id.nunique()==len(DEV)
        def obj(theta):
            b=calibrated_bin_probs(theta)
            residual=[]
            for row in dev_table.itertuples(index=False):
                count=hist1[f'{int(row.circle_id)}_count']
                residual.append(100*np.dot(count,b)/count.sum()-row.observed_fraction)
            return float(np.mean(np.square(residual))+2*((theta[0]-1)**2+theta[1]**2))
        fit=minimize(obj,[1,0],method='L-BFGS-B',bounds=[(.25,3.),(-3,3)])
        assert fit.success,fit.message
        fitted=calibrated_bin_probs(fit.x)
        (OUT/'M3_calibration.json').write_text(json.dumps(dict(a=float(fit.x[0]),b=float(fit.x[1]),
           development=sorted(DEV),holdout=sorted(HOLDOUT)),indent=2))
        calibrated=[]
        for row in m1.itertuples(index=False):
            count=hist1[f'{int(row.circle_id)}_count'].astype(float)
            positives=hist1[f'{int(row.circle_id)}_positive'].astype(float)
            predicted=100*np.dot(count,fitted)/count.sum()
            brier=(np.dot(positives,(1-fitted)**2)+np.dot(count-positives,fitted**2))/count.sum()
            calibrated.append(dict(model='M3',circle_id=row.circle_id,site=row.site,split=row.split,
              observed_fraction=row.observed_fraction,predicted_fraction=predicted,
              bias_pp=predicted-row.observed_fraction,brier=brier))
        m3=pd.DataFrame(calibrated)
        hold=pd.concat([m1,m2,m3],ignore_index=True).query("split=='holdout'").copy()
        assert hold.groupby('model').circle_id.nunique().eq(len(HOLDOUT)).all()
        hold['illumination']=mode;hold['fold']=fold
        hold.to_csv(OUT/'cv_holdout_scores.csv',index=False)
        all_fold_scores.append(hold)
        print('SAVED',mode,'fold',fold,'holdout circles',len(HOLDOUT),flush=True)
        del model1,model2,m1,m2,m3,hist1,hist2
        gc.collect();torch.cuda.empty_cache()
print('All available fold scores saved.')


## 08 — Central result: pooled held-out observed-versus-predicted fractions (22 circles)

Each independent circle is tested in precisely one fold. (R^2\), MAE, RMSE, bias, predicted-vs-observed slope and intercept are reported for each model and illumination treatment. The number of independent samples is 22 circles, **not** thousands of pixels.


In [ ]:
from scipy.stats import linregress
from matplotlib.lines import Line2D

frames=[]
for mode in MODES:
    for fold in FOLDS:
        path=BASE_OUT/mode/f'fold_{fold}'/'cv_holdout_scores.csv'
        if path.exists():frames.append(pd.read_csv(path))
assert len(frames)==len(MODES)*len(FOLDS),'Not all fold scores complete; run Block 07'
H=pd.concat(frames,ignore_index=True)
assert H.groupby(['illumination','model']).circle_id.nunique().eq(22).all()
def stats(d):
    y=d.observed_fraction.to_numpy(float)
    pred=d.predicted_fraction.to_numpy(float)
    reg=linregress(y,pred)
    return pd.Series(dict(n_circles=len(d),r2=r2_score(y,pred),pearson_r=reg.rvalue,
      slope=reg.slope,intercept_pp=reg.intercept,
      MAE_pp=mean_absolute_error(y,pred),RMSE_pp=np.sqrt(mean_squared_error(y,pred)),
      bias_pp=np.mean(pred-y),brier=d.brier.mean()))
MET=H.groupby(['illumination','model']).apply(stats,include_groups=False).reset_index()
display(MET.round(3).sort_values('MAE_pp'))
MET.to_csv(BASE_OUT/'02D_cv_metrics.csv',index=False)
H.to_csv(BASE_OUT/'02D_cv_circle_predictions.csv',index=False)
fig,axs=plt.subplots(2,3,figsize=(14,9),sharex=True,sharey=True,constrained_layout=True)
sites=['Baego','MOS','Yangambi','Yoko']
colors=dict(zip(sites,['#c27e24','#357ebd','#468c50','#944b99']))
for i,mode in enumerate(MODES):
    for j,model in enumerate(['M1','M2','M3']):
        ax=axs[i,j];a=H[(H.illumination==mode)&(H.model==model)]
        ax.plot([0,100],[0,100],color='gray',ls='--',lw=1)
        for row in a.itertuples(index=False):
            ax.scatter(row.observed_fraction,row.predicted_fraction,c=colors[row.site],s=24)
        metric=MET[(MET.illumination==mode)&(MET.model==model)].iloc[0]
        ax.set(title=f'{model} | {mode}\nR²={metric.r2:.2f}, MAE={metric.MAE_pp:.1f} pp',
               xlim=(0,100),ylim=(0,100))
        ax.grid(alpha=.15)
for ax in axs[-1,:]:ax.set_xlabel('Observed Musanga fraction (%)')
for ax in axs[:,0]:ax.set_ylabel('Predicted fraction (%)')
fig.savefig(BASE_OUT/'02D_observed_predicted_22_circles.png',dpi=170)
display(fig);plt.close(fig)
print('Figures and metrics:',BASE_OUT)
